# 异常处理与调试

学习目标：能保留错误原因、解释异常传播与清理顺序，并通过调用栈和断点定位同步错误。

前置知识：函数调用、条件判断、类、继承和 JSON 文本的基本形状。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 文件使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/15-errors-and-debugging/。

1. [throw-and-catch.mjs](scripts/15-errors-and-debugging/throw-and-catch.mjs)：创建错误、抛出和按类型处理。
2. [propagation.mjs](scripts/15-errors-and-debugging/propagation.mjs)：同步传播、重新抛出与 finally。
3. [error-cause.mjs](scripts/15-errors-and-debugging/error-cause.mjs)：自定义错误及原因链。
4. [cleanup-error.mjs](scripts/15-errors-and-debugging/cleanup-error.mjs)：finally 覆盖错误的独立反例。
5. [preserve-errors.mjs](scripts/15-errors-and-debugging/preserve-errors.mjs)：聚合操作和清理错误。
6. [stack.mjs](scripts/15-errors-and-debugging/stack.mjs)：从调用栈定位业务函数。
7. [debugger.mjs](scripts/15-errors-and-debugging/debugger.mjs)：供断点观察的计算函数。

## 1 错误对象与主动抛出

Error 保存错误描述；new Error 只是创建对象，throw 才会中断当前路径并抛出值。JavaScript 允许抛出任何值，但业务代码使用 Error 或其子类，便于保存名称、消息和宿主提供的调用栈。

| 错误类型原文名 | 中文名称／含义 | 典型触发条件 |
| --- | --- | --- |
| Error | 通用错误 | 应用自行报告失败 |
| TypeError | 类型错误 | 操作不适用于当前值 |
| RangeError | 范围错误 | 值超出允许范围 |
| ReferenceError | 引用错误 | 无法解析名称或访问未初始化绑定 |
| SyntaxError | 语法错误 | 解析代码或 JSON 失败 |
| URIError | URI 编解码错误 | URI 函数遇到不符合要求的编码 |
| EvalError | 求值错误类型 | 为旧版本兼容保留，当前规范不主动使用 |
| AggregateError | 聚合错误 | 在 errors 中保存多个错误 |

异常类型与消息并非一回事。本例用 TypeError 区分输入类型，再用 RangeError 表达负数或非整数不符合计数条件；这是应用选择的校验约定。

[throw-and-catch.mjs](scripts/15-errors-and-debugging/throw-and-catch.mjs)：

```javascript
function requireCount(value) {
  if (typeof value !== "number") throw new TypeError("count 必须是 number");
  if (!Number.isInteger(value) || value < 0) throw new RangeError("count 必须是非负整数");
  return value;
}
const pending = new Error("尚未抛出");
console.log(pending.name, pending.message, requireCount(3));
for (const value of ["3", -1]) {
  try {
    requireCount(value);
  } catch (error) {
    if (!(error instanceof TypeError || error instanceof RangeError)) throw error;
    console.log(error.name, error.message);
  }
}

// 按本例输入运行，输出依次为：
// Error 尚未抛出 3
// TypeError count 必须是 number
// RangeError count 必须是非负整数
```

Step 1：运行本节示例。

```bash
node scripts/15-errors-and-debugging/throw-and-catch.mjs
```

## 2 try、catch、finally 与传播

try 包围可能失败的操作，catch 接住从该同步执行路径传播来的抛出值；被调用函数没有捕获时，错误沿调用者向外传播。catch 可以处理已知错误并返回替代结果，也可以 throw 原错误继续传播。不要把所有未知错误都换成成功值。

finally 在控制离开 try/catch 时执行，包括 return 和 throw。下面只记录语言层的控制顺序，不使用文件或外部服务。异步回调稍后执行时，不在已结束的同步 try 调用路径中；Promise 的拒绝处理在后续章节展开。

[propagation.mjs](scripts/15-errors-and-debugging/propagation.mjs)：

```javascript
const events = [];
function inner() { throw new RangeError("数量不足"); }
function middle() {
  try { inner(); }
  catch (error) { events.push("middle"); throw error; }
  finally { events.push("finally"); }
}
try { middle(); }
catch (error) {
  if (!(error instanceof RangeError) || error.message !== "数量不足") throw error;
  events.push("outer:" + error.name);
}
console.log(events.join(","));
function complete() {
  try { return "结果"; }
  finally { events.push("返回前清理"); }
}
console.log(complete(), events.at(-1));

// 按本例输入运行，输出依次为：
// middle,finally,outer:RangeError
// 结果 返回前清理
```

Step 1：运行本节示例。

```bash
node scripts/15-errors-and-debugging/propagation.mjs
```

## 3 自定义异常与 cause

自定义错误类继承 Error，构造时调用 super(message, options)，再设置明确的 name。cause 保存导致本次错误的原始原因；它可以是任意值，不保证一定是 Error，因此处理外部错误链时应先判断。

包装错误适合补充业务上下文，例如“加载学习配置失败”，同时保留解析器原始错误。不要仅拼接一条新字符串丢掉原错误。下面只将 JSON 解析错误包装为 ConfigError，其他异常继续抛出。JSON 转换规则在下一相关章节展开。

[error-cause.mjs](scripts/15-errors-and-debugging/error-cause.mjs)：

```javascript
class ConfigError extends Error {
  constructor(message, options) {
    super(message, options);
    this.name = "ConfigError";
  }
}
function loadConfig(text) {
  try { return JSON.parse(text); }
  catch (cause) {
    if (!(cause instanceof SyntaxError)) throw cause;
    throw new ConfigError("学习配置不是有效 JSON", { cause });
  }
}
try { loadConfig('{"title":}'); }
catch (error) {
  if (!(error instanceof ConfigError) || !(error.cause instanceof SyntaxError)) throw error;
  console.log(error.name, error.message, error.cause.name);
  console.log(error instanceof Error);
}

// 按本例输入运行，输出依次为：
// ConfigError 学习配置不是有效 JSON SyntaxError
// true
```

Step 1：运行本节示例。

```bash
node scripts/15-errors-and-debugging/error-cause.mjs
```

## 4 清理失败不能覆盖原始原因

如果 finally 自己 return 或 throw，它会替换此前准备返回的值或传播的异常。清理代码不应随意 return。独立反例让两处都失败，观察最终传播的是清理异常。

如果应用必须保留操作失败和清理失败，可以捕获两者并用 AggregateError 汇总；errors 的次序在本例由代码约定，第一项是操作失败，第二项是清理失败。真正管理文件、监听器等资源时，应根据对应宿主 API 清理；finally 不提供进程被强制终止时的清理保证。

[cleanup-error.mjs](scripts/15-errors-and-debugging/cleanup-error.mjs)：

```javascript
try {
  throw new Error("operation failed");
} finally {
  throw new Error("cleanup failed");
}

// 独立运行：退出状态为 1；诊断包含 Error: cleanup failed。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/15-errors-and-debugging/cleanup-error.mjs
```

[preserve-errors.mjs](scripts/15-errors-and-debugging/preserve-errors.mjs)：

```javascript
function execute(operation, cleanup) {
  const errors = [];
  try { operation(); } catch (error) { errors.push(error); }
  try { cleanup(); } catch (error) { errors.push(error); }
  if (errors.length === 1) throw errors[0];
  if (errors.length > 1) throw new AggregateError(errors, "操作和清理均失败");
}
try {
  execute(() => { throw new Error("operation failed"); },
          () => { throw new Error("cleanup failed"); });
} catch (error) {
  if (!(error instanceof AggregateError)) throw error;
  console.log(error.name, error.message);
  console.log(error.errors.map(item => item.message).join(","));
}

// 按本例输入运行，输出依次为：
// AggregateError 操作和清理均失败
// operation failed,cleanup failed
```

Step 1：运行本节示例。

```bash
node scripts/15-errors-and-debugging/preserve-errors.mjs
```

## 5 阅读调用栈与宿主差异

Node.js 的 error.stack 由 V8 提供，记录 Error 创建位置及可用的调用帧，不是 ECMA-262 保证的统一文本格式。先看 name、message，再定位首个自己的函数和源码行列；不要把 Node 内部加载器帧当作业务出错点。

同一错误在不同宿主、版本或源码布局中的消息措辞、行号可能不同。业务分支优先依赖明确的错误类型或稳定的应用字段；Node 系统错误可在有文档保证时使用 code，不以整段错误字符串做通用协议。

[stack.mjs](scripts/15-errors-and-debugging/stack.mjs)：

```javascript
function parseCount() { throw new RangeError("count 超出范围"); }
function loadLesson() { return parseCount(); }
try { loadLesson(); }
catch (error) {
  if (!(error instanceof RangeError)) throw error;
  console.log(error.name, error.message);
  const frames = error.stack.split("\n");
  console.log(frames.some(line => line.includes("at parseCount")));
  console.log(frames.some(line => line.includes("at loadLesson")));
}

// 按本例输入运行，输出依次为：
// RangeError count 超出范围
// true
// true
```

Step 1：运行本节示例。

```bash
node scripts/15-errors-and-debugging/stack.mjs
```

## 6 使用断点和调试器

debugger 是语言语句：连接调试器时可在此暂停，未连接调试器时不会输出诊断。断点用于观察暂停位置的变量和调用栈，单步执行用于检查值在哪一步改变；运行成功不等于逻辑一定正确。

下面 multiply 的 unitPrice 表示单价，quantity 表示数量，total 表示乘积。普通执行先确认脚本可以结束，再在调试器里检查同一段代码。

[debugger.mjs](scripts/15-errors-and-debugging/debugger.mjs)：

```javascript
function multiply(unitPrice, quantity) {
  const total = unitPrice * quantity;
  debugger;
  return total;
}
console.log(multiply(12, 3));

// 按本例输入运行，输出依次为：
// 36
```

Step 1：运行本节示例。

```bash
node scripts/15-errors-and-debugging/debugger.mjs
```

## 7 在 Node.js 调试提示符中检查变量

以下操作使用 Node.js 24.11.0 自带的命令行调试器，工作目录仍为 content/编程语言/javascript。

Step 1：启动调试器并在首条可执行语句暂停。

```bash
node inspect scripts/15-errors-and-debugging/debugger.mjs
```

Step 2：在 debug&gt; 提示符输入 cont，运行到函数内的 debugger；输入 exec quantity 与 exec total，分别检查数量和乘积。

Step 3：输入 backtrace 查看 multiply 及调用位置；输入 next 执行到下一行，或输入 step 进入调用、out 返回调用者。

Step 4：输入 cont 让计算完成，然后在 debug&gt; 输入 .exit 退出调试器；不要遗留调试进程。

也可以在暂停处用 setBreakpoint 设置断点，用 clearBreakpoint 删除；用 watch('quantity') 持续观察表达式，使用 unwatch('quantity') 撤销观察。断点位置和命令回显由调试宿主决定，计算结果以配套源码的输入为准。

## 本章小结

- 创建 Error 不等于抛出；catch 处理已知失败，未知失败保留传播。
- cause 保留上下文关系，AggregateError 可保留多个失败；finally 也可能覆盖原异常。
- 调用栈和调试器属于宿主能力，定位自己的源码比依赖固定错误措辞更可靠。

## 练习

1. 给 ConfigError 增加文件逻辑名称字段，仍保留 cause。可核对标准：捕获对象同时含该名称和 SyntaxError 原因，正常 JSON 输入不抛出。
2. 修改 preserve-errors.mjs，使操作成功、仅清理失败。可核对标准：得到清理时抛出的原 Error；两处成功时正常退出。
3. 将 debugger.mjs 的输入改为单价 15、数量 4，在断点处读取 total。可核对标准：total 为 60，程序完成后调试器已退出。

## 参考与引用来源

- TC39 官方 ECMAScript 2025 分页版：[§20.5 错误类、cause 和 AggregateError](https://tc39.es/ecma262/2025/multipage/fundamental-objects.html#sec-error-objects)；[§14.14 throw](https://tc39.es/ecma262/2025/multipage/ecmascript-language-statements-and-declarations.html#sec-throw-statement)；[§14.15 try/catch/finally](https://tc39.es/ecma262/2025/multipage/ecmascript-language-statements-and-declarations.html#sec-try-statement)；[§14.16 debugger](https://tc39.es/ecma262/2025/multipage/ecmascript-language-statements-and-declarations.html#sec-debugger-statement)；[§25.5.1 JSON 解析异常](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-json.parse)。
- Node.js 官方文档：[24.11.0 error.stack、error.code 与 cause](https://nodejs.org/download/release/v24.11.0/docs/api/errors.html#errorstack)；[24.11.0 调试器：Stepping、Breakpoints、Information、Execution control](https://nodejs.org/download/release/v24.11.0/docs/api/debugger.html)。